# HR-only CNN sleep-stage classifier

This notebook is a comparison baseline for the multimodal CNN. It uses exactly the same six-block Conv1d architecture, context windows, optimizer, learning-rate schedule, early stopping, labels, and metrics, but uses only the 1 Hz Polar heart-rate signal. The FDC data is not loaded or used.

In [1]:
from __future__ import annotations

import copy
import math
import re
from pathlib import Path

import numpy as np
from IPython.display import display
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATASET_DIR = Path.cwd()
if DATASET_DIR.name != "datasets_for_training":
    raise RuntimeError("Open and run this notebook with datasets_for_training as the working directory.")
VALIDATION_DATE = "2026-10-17"
SAMPLE_RATE = 1
EPOCH_SECONDS = 30
CONTEXT_BEFORE = 5
CONTEXT_AFTER = 4
CONTEXT_EPOCHS = CONTEXT_BEFORE + 1 + CONTEXT_AFTER
SAMPLES_PER_EPOCH = EPOCH_SECONDS * SAMPLE_RATE
INPUT_SAMPLES = CONTEXT_EPOCHS * SAMPLES_PER_EPOCH
BATCH_SIZE = 100
LEARNING_RATE = 1e-4
DROPOUT = 0.1
MAX_EPOCHS = 300
LR_PATIENCE = 10
EARLY_STOPPING_PATIENCE = 30
STAGES = ["Awake", "REM", "Core", "Deep"]
STAGE_TO_INDEX = {stage: index for index, stage in enumerate(STAGES)}
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required by this notebook, but torch.cuda.is_available() is False.")
DEVICE = torch.device("cuda")
print("Dataset:", DATASET_DIR)
print("Device:", torch.cuda.get_device_name(DEVICE))
print("PyTorch:", torch.__version__)
print("Validation sleep:", VALIDATION_DATE)

Dataset: c:\BME\Szemfedo\CapMeter\data_analysis\sleep_tests\datasets_for_training
Device: NVIDIA GeForce RTX 3050 6GB Laptop GPU
PyTorch: 2.11.0+cu128
Validation sleep: 2026-10-17


## Loading and aligning the training files

Only the date-matched Apple Watch and Polar files are required. Apple Watch stages are labels; the Polar heart rate is the sole input signal.

In [2]:
def _date_from_name(path: Path) -> str:
    match = re.fullmatch(r"(\d{4}-\d{2}-\d{2})", path.stem)
    if match is None:
        raise ValueError(f"Unexpected Apple Watch filename: {path.name}")
    return match.group(1)


def _find_files(recording_date: str) -> tuple[Path, Path]:
    apple = DATASET_DIR / f"{recording_date}.csv"
    polar_files = sorted(DATASET_DIR.glob(f"polar_{recording_date}_*.CSV"))
    if not apple.is_file() or not polar_files:
        raise FileNotFoundError(f"Missing Apple Watch or Polar file for {recording_date}")
    return apple, polar_files[0]


def _load_sleep(recording_date: str) -> pd.DataFrame:
    apple_path, polar_path = _find_files(recording_date)
    apple = pd.read_csv(apple_path)
    for column in ("Start time", "End time"):
        apple[column] = pd.to_datetime(apple[column]).map(
            lambda value: value.tz_localize(None) if value.tzinfo else value
        )
    apple_start = apple["Start time"].min()
    apple["start_s"] = (apple["Start time"] - apple_start).dt.total_seconds()
    apple["end_s"] = (apple["End time"] - apple_start).dt.total_seconds()

    polar = pd.read_csv(polar_path)
    polar["time_s"] = pd.to_timedelta(polar["Time"]).dt.total_seconds()
    polar["time_s"] -= polar["time_s"].min()
    polar["HR (bpm)"] = pd.to_numeric(polar["HR (bpm)"], errors="coerce")
    polar = polar.dropna(subset=["time_s", "HR (bpm)"]).drop_duplicates("time_s")

    duration = min(float(apple["end_s"].max()), float(polar["time_s"].max()))
    seconds = np.arange(0, math.floor(duration) + 1, dtype=np.float32)
    result = pd.DataFrame({"time_s": seconds})
    result["HR (bpm)"] = np.interp(seconds, polar["time_s"], polar["HR (bpm)"])
    result["stage"] = pd.NA
    for row in apple.itertuples(name=None, index=False):
        mask = (result["time_s"] >= row[3]) & (result["time_s"] < row[4])
        result.loc[mask, "stage"] = row[0]
    return result.dropna(subset=["stage"]).reset_index(drop=True)


recording_dates = sorted(_date_from_name(path) for path in DATASET_DIR.glob("20??-??-??.csv"))
if len(recording_dates) != 4 or VALIDATION_DATE not in recording_dates:
    raise ValueError(f"Expected four sleeps including {VALIDATION_DATE}; found {recording_dates}")
sleep_data = {recording_date: _load_sleep(recording_date) for recording_date in recording_dates}
for recording_date, data in sleep_data.items():
    print(recording_date, len(data), "one-Hz rows", data["stage"].value_counts().to_dict())

2026-10-16 21445 one-Hz rows {'Core': 11420, 'REM': 5373, 'Deep': 3931, 'Awake': 721}
2026-10-17 21094 one-Hz rows {'Core': 8882, 'Deep': 5401, 'REM': 5190, 'Awake': 1621}
2026-10-19 19868 one-Hz rows {'Core': 9812, 'Deep': 4651, 'REM': 4475, 'Awake': 930}
2026-10-20 8836 one-Hz rows {'Core': 5255, 'Deep': 2641, 'REM': 940}


## Context windows and labels

Each input contains 5 preceding epochs, the target epoch, and 4 following epochs. The 1 Hz heart-rate samples therefore form a 300-sample input sequence.

In [3]:
def _make_examples(data: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    usable_seconds = len(data) // EPOCH_SECONDS * EPOCH_SECONDS
    data = data.iloc[:usable_seconds].copy()
    values = data[["HR (bpm)"]].to_numpy(np.float32)
    labels = data["stage"].map(STAGE_TO_INDEX).to_numpy()
    epoch_labels = []
    for start in range(0, usable_seconds, EPOCH_SECONDS):
        block = labels[start:start + EPOCH_SECONDS]
        epoch_labels.append(int(np.bincount(block, minlength=len(STAGES)).argmax()))
    examples, targets = [], []
    for center in range(CONTEXT_BEFORE, len(epoch_labels) - CONTEXT_AFTER):
        start = (center - CONTEXT_BEFORE) * EPOCH_SECONDS
        stop = (center + CONTEXT_AFTER + 1) * EPOCH_SECONDS
        examples.append(values[start:stop])
        targets.append(epoch_labels[center])
    return np.asarray(examples, dtype=np.float32), np.asarray(targets, dtype=np.int64)


def _standardize(train_x: np.ndarray, other_x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mean = train_x.reshape(-1, train_x.shape[-1]).mean(axis=0)
    std = train_x.reshape(-1, train_x.shape[-1]).std(axis=0)
    std[std < 1e-6] = 1.0
    return (train_x - mean) / std, (other_x - mean) / std


train_parts = [_make_examples(sleep_data[d]) for d in recording_dates if d != VALIDATION_DATE]
validation_x, validation_y = _make_examples(sleep_data[VALIDATION_DATE])
train_x = np.concatenate([part[0] for part in train_parts])
train_y = np.concatenate([part[1] for part in train_parts])
train_x, validation_x = _standardize(train_x, validation_x)
train_x = torch.from_numpy(train_x.transpose(0, 2, 1))
validation_x = torch.from_numpy(validation_x.transpose(0, 2, 1))
train_y = torch.from_numpy(train_y)
validation_y = torch.from_numpy(validation_y)
print("Train:", tuple(train_x.shape), tuple(train_y.shape))
print("Validation:", tuple(validation_x.shape), tuple(validation_y.shape))

Train: (1643, 1, 300) (1643,)
Validation: (694, 1, 300) (694,)


In [4]:
class SleepCNN(nn.Module):
    def __init__(self, channels: int = 1, classes: int = 4) -> None:
        super().__init__()
        blocks = []
        in_channels = channels
        for _ in range(6):
            blocks.extend([
                nn.Conv1d(in_channels, 64, kernel_size=5, padding=2),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.MaxPool1d(2),
                nn.Dropout(DROPOUT),
            ])
            in_channels = 64
        self.features = nn.Sequential(*blocks)
        with torch.no_grad():
            feature_size = self.features(torch.zeros(1, channels, INPUT_SAMPLES)).flatten(1).shape[1]
        self.classifier = nn.Linear(feature_size, classes)
        self.apply(self._initialize)

    @staticmethod
    def _initialize(module: nn.Module) -> None:
        if isinstance(module, (nn.Conv1d, nn.Linear)):
            nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x).flatten(1))


model = SleepCNN().to(DEVICE)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=LR_PATIENCE)
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
validation_loader = DataLoader(TensorDataset(validation_x, validation_y), batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
print(model)

SleepCNN(
  (features): Sequential(
    (0): Conv1d(1, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.1, inplace=False)
    (5): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU()
    (8): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Dropout(p=0.1, inplace=False)
    (10): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (11): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU()
    (13): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Dropout(p=0.1, inplace=False)
    (15): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,))

In [5]:
def _evaluate(loader: DataLoader) -> tuple[float, np.ndarray]:
    model.eval()
    total_loss, predictions, targets = 0.0, [], []
    with torch.no_grad():
        for features, labels in loader:
            features = features.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = model(features)
            total_loss += loss_function(logits, labels).item() * labels.size(0)
            predictions.append(logits.argmax(1).cpu().numpy())
            targets.append(labels.cpu().numpy())
    return total_loss / len(loader.dataset), np.column_stack((np.concatenate(targets), np.concatenate(predictions)))


history, best_state, best_loss, stale = [], None, float("inf"), 0
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    for features, labels in train_loader:
        features = features.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss_function(model(features), labels).backward()
        optimizer.step()
    validation_loss, _ = _evaluate(validation_loader)
    scheduler.step(validation_loss)
    history.append((epoch, validation_loss, optimizer.param_groups[0]["lr"]))
    if validation_loss < best_loss - 1e-6:
        best_loss, best_state, stale = validation_loss, copy.deepcopy(model.state_dict()), 0
    else:
        stale += 1
    if epoch == 1 or epoch % 10 == 0:
        print(f"epoch={epoch:03d} validation_loss={validation_loss:.5f} lr={optimizer.param_groups[0]['lr']:.2e}")
    if stale >= EARLY_STOPPING_PATIENCE:
        print("Early stopping at epoch", epoch)
        break
model.load_state_dict(best_state)
print("Best validation loss:", best_loss)

epoch=001 validation_loss=1.32956 lr=1.00e-04
epoch=010 validation_loss=1.00681 lr=1.00e-04
epoch=020 validation_loss=0.94226 lr=1.00e-04
epoch=030 validation_loss=0.93072 lr=1.00e-04
epoch=040 validation_loss=0.92425 lr=1.00e-04
epoch=050 validation_loss=0.93430 lr=1.00e-04
epoch=060 validation_loss=0.92401 lr=5.00e-05
epoch=070 validation_loss=0.92558 lr=2.50e-05
Early stopping at epoch 78
Best validation loss: 0.9039680385795725


In [6]:
def classification_report(targets: np.ndarray, predictions: np.ndarray) -> pd.DataFrame:
    matrix = np.zeros((len(STAGES), len(STAGES)), dtype=np.int64)
    for target, prediction in zip(targets, predictions):
        matrix[target, prediction] += 1
    rows = []
    for index, stage in enumerate(STAGES):
        tp = matrix[index, index]
        precision = tp / matrix[:, index].sum() if matrix[:, index].sum() else 0.0
        recall = tp / matrix[index].sum() if matrix[index].sum() else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({"stage": stage, "precision": precision, "recall": recall, "f1": f1, "support": matrix[index].sum()})
    total = matrix.sum()
    accuracy = np.trace(matrix) / total if total else 0.0
    expected = (matrix.sum(axis=0) * matrix.sum(axis=1)).sum() / total**2 if total else 0.0
    kappa = (accuracy - expected) / (1 - expected) if expected < 1 else 0.0
    report = pd.DataFrame(rows).set_index("stage")
    print("Accuracy:", accuracy, "Cohen kappa:", kappa)
    print("Confusion matrix (rows=true, columns=predicted):")
    display(pd.DataFrame(matrix, index=STAGES, columns=STAGES))
    return report


validation_loss, result = _evaluate(validation_loader)
report = classification_report(result[:, 0], result[:, 1])
display(report)
print("TST ground truth (hours):", (result[:, 0] != STAGE_TO_INDEX["Awake"]).sum() * EPOCH_SECONDS / 3600)
print("TST CNN (hours):", (result[:, 1] != STAGE_TO_INDEX["Awake"]).sum() * EPOCH_SECONDS / 3600)

Accuracy: 0.659942363112392 Cohen kappa: 0.5101523222184671
Confusion matrix (rows=true, columns=predicted):


,Awake,REM,Core,Deep
Awake,11,10,28,5
REM,8,113,35,17
Core,1,34,181,71
Deep,11,4,12,153


,precision,recall,f1,support
stage,,,,
Awake,0.354839,0.203704,0.258824,54
REM,0.701863,0.653179,0.676647,173
Core,0.707031,0.630662,0.666667,287
Deep,0.621951,0.850000,0.718310,180


TST ground truth (hours): 5.333333333333333
TST CNN (hours): 5.525
